In [2]:
import scanpy as sc
import rapids_singlecell as rsc
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from tqdm import tqdm

sc.set_figure_params(frameon=False, dpi=80, fontsize=12)

from scatlastb_utils.io import read_anndata, write_zarr_linked
from scatlastb_utils.utils import dask_compute
from scatlastb_utils.metrics import compare_distances, plot_distances_scatter, plot_ranked_distances

# Read Data

In [14]:
ll data/pipeline/HLCAv1/QC/collect/dataset~highQC.zarr/layers/

total 0


In [15]:
adata_all = read_anndata(
    'data/pipeline/HLCAv1/QC/collect/dataset~highQC.zarr/',
    # X='layers/normcounts',
    obs='obs',
    var='var',
    obsm='obsm',
    uns='uns',
    obsp='obsp',
    dask_slots=['layers', 'X', 'obsp'],
    dask=True,
    backed=True,
)

dask: True, backed: True
chunks: (200000, -1)
Read slot "obs", store as "obs"...
Read slot "var", store as "var"...
Read slot "obsm", store as "obsm"...


Read obsm slots as_dask=False: 100%|██████████| 2/2 [00:00<00:00,  7.24it/s]

Read slot "uns", store as "uns"...



Read uns slots as_dask=False: 100%|██████████| 4/4 [00:00<00:00, 37.43it/s]

Read slot "obsp", store as "obsp"...



Read obsp slots as_dask=True: 100%|██████████| 2/2 [00:00<00:00, 95.28it/s]

shape: (584944, 27957)


## Clean slot names

In [ ]:
def clean_slot_names(adata, replace_str='', replace_with=''):
    for name, slot in adata.__dict__.items():
        if isinstance(slot, dict):
            slot = {k.replace(replace_str, replace_with): v for k, v in slot.items()}
            setattr(adata, name, slot)
        elif isinstance(slot, pd.DataFrame):
            slot.columns = [x.replace(replace_str, replace_with) for x in slot.columns]
            # setattr(adata, name, slot)

In [ ]:
clean_slot_names(
    adata_all,
    replace_str='marker_genes:label_transfer:clustering:filter:integration:preprocessing:'
)

In [ ]:
clean_slot_names(
    adata_qc,
    replace_str='marker_genes:label_transfer:clustering:integration:preprocessing:filter:'
)

In [ ]:
adata = adata_qc

In [ ]:
# collect all outputs into single object
indices = adata_all.obs_names.isin(adata.obs_names)

for key, value in tqdm(adata_all.obsm.items()):
    # adata.obsm[key] = value[indices, :]
    adata.obsm[key] = value

for key, value in tqdm(adata_all.obsp.items()):
    # adata.obsp[key] = value.obsp[key]
    adata.obsp[key] = value

for key in tqdm(adata_all.obs.columns):
    # adata.obs[key] = adata_all.obs[indices][key]
    adata.obs[key] = adata_all.obs[key]

for key, value in tqdm(adata_all.uns.items()):
    adata.uns[key] = value

In [ ]:
# set colors
ref_key = 'manual_ann'
for col in tqdm([x for x in adata.obs.columns if x.startswith('majority_reference')]):
    adata.obs[col] = adata.obs[col].cat.set_categories(adata.obs[ref_key].cat.categories)

# Graph dissimilarity

In [ ]:
integration_alias = {
    'unintegrated': '530a81710b',
    'harmony': '2e34347366',
    'scvi': '13f239d804',
    'scanvi': '530a81710b',
}

In [ ]:
method = 'scvi'

int1 = 'all'
int2 = 'qc'

integration1 = f'all--integration={integration_alias[method]}'
integration2 = f'qc--integration={integration_alias[method]}'

In [ ]:
adata.obsp

In [ ]:
%%time
comp = f'{int1}-vs-{int2}'

adata.obs[[
    f'avg_distance_1:{comp}',
    f'avg_distance_2:{comp}',
    f'avg_difference:{comp}',
    f'avg_distance_diff:{comp}',
    f'spearman_correlation:{comp}',
]] = compare_distances(
    adata,
    obsp_key_1=f'distances--{integration1}',
    obsm_key_1=f'X_emb--{integration1}',
    obsp_key_2=f'distances--{integration2}',
    obsm_key_2=f'X_emb--{integration2}',
    k_max=100,
    scale_distances=True,
    log_scale_diffs=False,
)

comp = f'{int2}-vs-{int1}'
adata.obs[[
    f'avg_distance_1:{comp}',
    f'avg_distance_2:{comp}',
    f'avg_difference:{comp}',
    f'avg_distance_diff:{comp}',
    f'spearman_correlation:{comp}',
]] = compare_distances(
    adata,
    obsp_key_1=f'distances--{integration2}',
    obsm_key_1=f'X_emb--{integration2}',
    obsp_key_2=f'distances--{integration1}',
    obsm_key_2=f'X_emb--{integration1}',
    k_max=100,
    scale_distances=True,
    log_scale_diffs=False,
)

In [ ]:
plot_ranked_distances(adata, int1, int2)
plot_distances_scatter(adata, int1, int2, s=1, alpha=0.5)
plot_distances_scatter(adata, int2, int1, s=1, alpha=0.5)

In [ ]:
comparisons = [
    f'{int1}-vs-{int2}',
    f'{int2}-vs-{int1}'
]
metrics = [
    # 'avg_distance_1',
    # 'avg_distance_2',
    # 'avg_difference',
    'avg_distance_diff',
    # 'spearman_correlation'
]

In [ ]:
for _metric in metrics:
    colors = [f'{_metric}:{comparison}' for comparison in comparisons]
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration1}',
        color=colors,
        cmap='viridis',
        # size=5,
        ncols=len(metrics),
        wspace=0.5,
        # vmin=0,
        vmax=adata.obs[colors].quantile(0.99).tolist(),
        sort_order=True,
    )
 
    sc.pl.embedding(
        adata,
        basis=f'X_umap--{integration2}',
        color=colors,
        cmap='viridis',
        # size=5,
        ncols=len(metrics),
        wspace=0.5,
        # vmin=0,
        vmax=adata.obs[colors].quantile(0.99).tolist(),
        sort_order=True,
    )

## Determine clusters of interest

In [ ]:
def plot_diff_per_clusters(
    adata,
    cluster_key,
    diff_key,
    x_label=None,
    width=6,
    height_factor=0.2,
    figsize=None,
    agg='mean',
):
    if x_label is None:
        x_label = diff_key
    
    value_counts = adata.obs[cluster_key].astype(str).value_counts()
    diff_per_cluster = adata.obs.groupby(
        cluster_key,
        observed=True
    )[diff_key].agg(agg)
    diff_per_cluster = diff_per_cluster.sort_values()

    # plot
    if figsize is None:
        figsize = (width, len(diff_per_cluster) * height_factor)
    fig, ax = plt.subplots(figsize=figsize)
    
    diff_per_cluster.plot.barh(ax=ax)
    
    for i, (k, v) in enumerate(diff_per_cluster.items()):
        ax.text(v, i, f'{value_counts[k]}', va='center')
    
    ax.grid(False)
    ax.set_xlabel(x_label)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.show()
    
    return diff_per_cluster

In [ ]:
plot_diff_per_clusters(
    adata,
    'original_ann_level_1',
    f'avg_distance_diff:{int1}-vs-{int2}',
)

In [ ]:
plot_diff_per_clusters(
    adata,
    'manual_ann',
    f'avg_distance_diff:{int1}-vs-{int2}',
)

In [ ]:
plot_diff_per_clusters(
    adata,
    f'leiden_0.5_1--{integration1}',
    f'avg_distance_diff:{int1}-vs-{int2}',
)

In [ ]:
plot_diff_per_clusters(
    adata,
    f'leiden_0.5_1--{integration2}',
    f'avg_distance_diff:{int1}-vs-{int2}',
)

In [ ]:
colors = [
    'manual_ann',
    f'majority_reference--{integration1}',
    f'majority_reference--{integration2}',
    f'leiden_0.5_1--{integration1}',
    f'leiden_0.5_1--{integration2}',
]

In [ ]:
sc.pl.embedding(
    adata,
    basis=f'X_umap--{integration1}',
    color=colors,
    ncols=1,
)

In [ ]:
sc.pl.embedding(
    adata,
    basis=f'X_umap--{integration2}',
    color=colors,
    ncols=1,
)

## Subset to Epithelial lineage

In [ ]:
epi_mask = adata.obs['original_ann_level_1'] == 'Epithelial'

In [ ]:
colors

In [ ]:
sc.pl.embedding(
    adata,
    basis=f'X_umap--{integration1}',
    color='manual_ann',
    mask_obs=epi_mask,
    ncols=1,
)

In [ ]:
adata.obs['manual_ann_epi'] = adata.obs[epi_mask]['manual_ann']
adata.obs['manual_ann_epi'] = adata.obs['manual_ann_epi'].cat.remove_unused_categories()

In [ ]:
sc.pl.embedding(
    adata,
    basis=f'X_umap--{integration1}',
    color='manual_ann_epi',
    ncols=1,
)

In [ ]:
adata_epi = adata[epi_mask].copy()

In [ ]:
def recompute_knn(adata, integration):
    rsc.pp.neighbors(adata, use_rep=f'X_emb--{integration}', n_neighbors=100)
    rsc.tl.umap(
        adata,
        min_dist=0.5,
        spread=1.0,
        n_components=2,
        maxiter=None,
        alpha=1.0,
        # gamma=1.0,
        negative_sample_rate=5
    )
    adata.obsp[f'connectivities--{integration}'] = adata.obsp['connectivities']
    adata.obsp[f'distances--{integration}'] = adata.obsp['distances']
    adata.obsm[f'X_umap--{integration}'] = adata.obsm['X_umap']

In [ ]:
recompute_knn(adata_epi, integration1)
recompute_knn(adata_epi, integration2)

In [ ]:
%%time
write_zarr_linked(
    adata_epi,
    in_dir='data/pipeline/qc/collect/dataset~HCA_integration_filtered.zarr',
    out_dir='data/output/qc/hlca_epithelial.zarr',
    subset_mask=(epi_mask, np.full(adata.n_vars, True)),
    files_to_keep=['obs', 'obsm', 'obsp', 'uns'],
)

In [ ]:
colors = [
    'manual_ann',
    f'leiden_0.5_1--{integration1}',
    f'leiden_0.5_1--{integration2}',
    'scrublet_score',
    'n_genes',
    'n_counts',
    'percent_mito',
    # f'leiden_0.5_2--{integration1}',
    # f'leiden_0.5_2--{integration2}',
    'study',
]

In [ ]:
for col in colors: 
    sc.pl.embedding(
        adata_epi,
        basis=f'X_umap--{integration1}',
        color=col,
        ncols=1,
    )

    sc.pl.embedding(
        adata_epi,
        basis=f'X_umap--{integration2}',
        color=col,
        ncols=1,
    )